# Transfer budget sweep (llm_transfer.sweep): accuracy as a function of the thinking budget

At the fixed 6k cap every model was cut off on 92-100% of the bridge's 3-4 jump backward traps, so the bridge could not
tell "the skill did not transfer" from "the text format costs more tokens". Here each item is generated ONCE at 24k and
scored at 2k / 4k / 6k / 8k / 12k / 16k / 24k (finished within B; forced from the rollout cut at B). A curve shifted
left is efficiency; a higher plateau is skill.

Bridge first (200 items). Models run in priority order: `grpo` and `r14` answer the main question (does GRPO's in-domain
gain on backward traps survive the change of surface?); stop after them if time is short. Raw generations are written to
Drive as they arrive, so a disconnect loses only the requests in flight: rerun the cells and it resumes.
**Runtime:** A100. ~45-75 min per model (most bridge items run long).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, subprocess, time, urllib.request
OUT = '/content/drive/MyDrive/phutball/transfer'; SWEEP = f'{OUT}/sweep'; os.makedirs(SWEEP, exist_ok=True)
STATE = '/content/drive/MyDrive/phutball/grpo_cost/gate_state.json'
MODELS = {                                                 # priority order; name -> LoRA adapter (None: base model)
    'grpo': '/content/drive/MyDrive/phutball/grpo_cost/p11/final',   # SAME adapter as the v1 transfer run's 'grpo'
    'r14': '/content/drive/MyDrive/phutball/curriculum_v4/r14/adapter/final',
    'gbest': json.load(open(STATE))['best'],               # the current gated-GRPO best (p12 or later)
    'base': None,
    'warm': '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final',
}
FAMILIES = 'bridge'
CUTS = '2048,4096,6144,8192,12288,16384,24576'; MAXLEN = 26624
print(MODELS['gbest'])
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao


In [ ]:
%cd /content/phutball-jax
# the frozen v1 suite (the bridge items are the same in every build; frozen anyway so ids and prompts match the v1 runs)
import shutil
if os.path.exists(f'{OUT}/suite_v1.jsonl'):
    shutil.copy(f'{OUT}/suite_v1.jsonl', 'llm_transfer/data/suite_v1.jsonl'); print('frozen suite from Drive')
else:
    subprocess.run(['python', '-m', 'llm_transfer.build', '--out', 'llm_transfer/data/suite_v1.jsonl'], check=True)
    shutil.copy('llm_transfer/data/suite_v1.jsonl', f'{OUT}/suite_v1.jsonl')
adapters = {k: v for k, v in MODELS.items() if v}
!pkill -f "vllm serve" ; sleep 5
log = open('/content/vllm_serve.log', 'w')
p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(MAXLEN), '--gpu-memory-utilization', '0.90',
                      '--port', '8000', '--enable-lora', '--max-lora-rank', '32', '--max-loras', str(len(adapters)),
                      '--lora-modules', *[f'{k}={v}' for k, v in adapters.items()]], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try: urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up'); break
    except Exception: time.sleep(5)
else: raise RuntimeError(open('/content/vllm_serve.log').read()[-2000:])


In [ ]:
for name in MODELS:                                        # resumable: finished models are skipped, partial ones resume
    if os.path.exists(f'{SWEEP}/{name}_sweep.jsonl'): print('done:', name); continue
    model = 'Qwen/Qwen3.5-9B' if MODELS[name] is None else name
    !python -m llm_transfer.sweep --model {model} --name {name} --families {FAMILIES} --cuts {CUTS} --gen-dir {SWEEP} --concurrency 128 | tail -25
    !cp llm_transfer/results/{name}_sweep.jsonl {SWEEP}/


In [ ]:
!mkdir -p llm_transfer/results && cp {SWEEP}/*_sweep.jsonl llm_transfer/results/
names = ' '.join(n for n in MODELS if os.path.exists(f'{SWEEP}/{n}_sweep.jsonl'))
!python -m llm_transfer.sweep_report {names}


In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()
